In [ ]:
import google.generativeai as genai
import os
import fitz  # PyMuPDF
import json
import re
import time
from dotenv import load_dotenv

# --- Configuration ---
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")

MODEL_NAME = 'gemma-4-31b-it'

# Folder structure
PDF_FOLDER = "pdfs"
JSON_FOLDER = "jsons_test"

# Rate limit: 10 requests per minute → 6 seconds delay
REQUEST_DELAY_SECONDS = 6

API_RETRY_ATTEMPTS = 3
API_RETRY_DELAY_SECONDS = 10

#Note that this is a example system instruction. You should customize it based on your specific research questions and extraction needs. The more detailed and clear the instruction, the better the model's output will align with your requirements.
SYSTEM_INSTRUCTION = """You are an expert research assistant performing a systematic literature review on wearable biometric authentication.

Your task is to:
1. Determine whether the paper is relevant to the review
2. Extract structured data ONLY if relevant
3. Output a clean, valid JSON object
4. Base extraction on abstract, method, evaluation, and conclusion sections. Crosscheck across sections.

=========================
STEP 1 — RELEVANCE FILTER
=========================

A paper is RELEVANT if it satisfies ALL:
- Focuses on biometric authentication or identification
- Uses wearable, on-body, OR mobile sensors/devices (e.g., smartphones, smartwatches, headsets)
- Involves physiological or behavioral signals

EXCLUDE if:
- Not about authentication (e.g., only activity recognition, gesture control, health monitoring)
- Not biometric (e.g., password systems, PIN-based, token-based)
- Not wearable/on-body/mobile (e.g., CCTV face recognition, stationary wall cameras)
- Purely theoretical with no biometric system implemented or evaluated

Set:
"relevance": "include | exclude | uncertain"
"exclusion_reason": ""  ← Leave as empty string "" if relevance = "include"

======================
SYNONYM MAPPING RULES
======================
CRITICAL: If the exact keyword is absent but a synonym exists, map it to the controlled term. Do NOT output null when a reasonable mapping exists. Apply consistently:

"Attack Success Rate", "ASR", "Imposter Accept Rate", "False Match Rate", "FMR" -> map to FAR
"False Non-Match Rate", "FNMR" -> map to FRR
"Area Under Curve", "AUROC" -> map to AUC

COLLECTION SETTING:
- "naturalistic", "free-living", "daily life", "uncontrolled" → "in-the-wild"
- "office environment", "partially controlled", "limited constraints" → "semi-controlled"
- "controlled", "laboratory", "lab setting", "simulated" → "lab"

AUTHENTICATION TYPE:
- "passive", "transparent", "zero-effort", "unobtrusive" → "implicit"
- "background", "always-on", "seamless", "ambient" → "continuous"
- "challenge-response", "prompted", "on-demand" → "explicit"

LEARNING TYPE:
- "end-to-end", "neural network-based", "CNN" → "supervised"
- "unlabeled data", "contrastive learning" → "self-supervised"
- "few labeled samples", "N-way K-shot", "Siamese" → "few-shot"
- "domain adaptation", "pre-trained model", "fine-tuning" → "transfer"
- "clustering", "autoencoder" without labels → "unsupervised"

FUSION LEVEL:
- "early fusion", "input-level", "raw signal concatenation" → "feature"
- "score-level", "late fusion", "probability fusion" → "score"
- "voting", "majority rule" → "decision"

=========================
GENERAL EXTRACTION RULES
=========================
- **TARGET METRICS ONLY:** In the evaluation section, ONLY extract AUC, EER, FRR, and FAR. Ignore Accuracy, Precision, Recall, F1, etc.
- **EXTRACT THE BEST RESULT:** Extract ONLY the absolute best reported result for each target metric. 
  * CRITICAL RULE FOR "BEST": For AUC, "best" means the HIGHEST value. For EER, FAR, and FRR, "best" means the LOWEST value.
- **CONTEXTUAL EVALUATION:** Include context for the best metric (e.g., "Achieved on Dataset 1 using Multimodal Feature Fusion").
- **FLOAT METRICS:** Metrics MUST allow decimals/floats (e.g., 1.25). Do NOT round to integers.
- Only extract explicitly stated info. If missing → use `null`. Do NOT guess, infer, or hallucinate.

=========================
SPECIFIC DEFINITIONS
=========================
REAL-WORLD GAP:
- `environments_compared`: true ONLY if they experimentally tested and compared lab data vs uncontrolled real-world data.
- `gap_acknowledged_in_text`: true if authors discuss degradation outside the lab, even if not tested.

PERFORMANCE DECAY & TEMPORAL ROBUSTNESS:
- `cross_session`: true if data collected across multiple days/sessions.
- `time_delta`: specify the time elapsed for the decay (e.g., "1 week", "6 months").

SECURITY BLOCK:
- `spoofing_tested`: true ONLY if the paper tests active presentation attacks (e.g., replay, synthetic, 3D masks). **Zero-effort or random imposter attempts do NOT count as spoofing.**
- `attack_types`: Use ["zero-effort", "replay", "presentation/mask", "synthetic/deepfake", "adversarial"].
- `defense_type`: Refers ONLY to dedicated anti-spoofing/Liveness mechanisms, NOT the main biometric matcher.
- `liveness_detection`: true (boolean) if the system actively checks if the user is alive/present.

GAPS & LIMITATIONS (Controlled Vocabularies):
- DATA LIMITATION: SMALL_SAMPLE_SIZE, NO_PUBLIC_DATASET, SINGLE_DATASET_ONLY
- EVALUATION GAPS: REAL_WORLD_MISSING, NO_CROSS_SESSION, NO_TEMPORAL_TEST, NO_BASELINE_COMPARISON
- SECURITY GAPS: NO_SPOOFING_TEST, LIMITED_ATTACK_TYPES, NO_ADVERSARIAL_DEFENSE
- USABILITY GAPS: NO_USER_STUDY, HIGH_LATENCY, HIGH_ENERGY_COST, USER_INTERACTION_REQUIRED
- MODEL LIMITATIONS: NO_ADAPTATION, NO_FUSION, HEAVY_MODEL
- ENVIRONMENT: LAB_ONLY_DATASET, SEMI_CONTROLLED_ONLY

===================
OUTPUT FORMAT
===================

IF relevance = "exclude", Return ONLY:
{
  "title": "",
  "relevance": "exclude",
  "exclusion_reason": ""
}

IF relevance = "include" OR "uncertain", Return FULL JSON:

{
  "title": "",
  "year": null,
  "venue": "",
  "doi": "",
  "relevance": "",
  "exclusion_reason": "",
  "rq_category": ["RQ1 |& RQ2 |& RQ3 |& RQ4 |& RQ5"],

  "biometric": {
    "type": "physiological | behavioral | hybrid | null",
    "modality": [],
    "sensor": [],
    "device": [],
    "body_location": ""
  },

  "data": [
    {
      "dataset_name": "",
      "dataset_type": "public | private | null",
      "dataset_role": "training | validation | testing | benchmark | combined | null",
      "num_subjects": null,
      "num_samples": null,
      "collection_setting": "lab | semi-controlled | in-the-wild | null",
      "duration": { "value": null, "unit": "" }
    }
  ],

  "method": {
    "learning_type": "supervised | unsupervised | semi-supervised | self-supervised | few-shot | transfer | contrastive | null",
    "model": [],
    "input_representation": "",
    "training_strategy": ""
  },

  "evaluation": [
    {
      "metric": "AUC | EER | FRR | FAR", 
      "value": null, 
      "context": "" 
    }
  ],
  "evaluation_protocol": "",
  "real_world_tested": null,

  "real_world_gap": {
    "environments_compared": null,
    "gap_acknowledged_in_text": null,
    "lab_performance_value": null,
    "real_world_performance_value": null,
    "drop_percentage": null
  },

  "temporal_robustness": {
    "cross_session_tested": null,
    "cross_day_tested": null,
    "longitudinal_duration": { "value": null, "unit": "" },
    "performance_decay_over_time": {
      "is_quantified": null,
      "time_delta": "{value: null, unit: ''}",
      "drop_in_performance": { "metric": "", "value": null }
    }
  },

  "performance_analysis": {
    "robust_to_motion": { "is_quantified": null, "metric": "", "value": null },
    "robust_to_noise": { "is_quantified": null, "metric": "", "value": null },
    "reported_limitations": []
  },

  "security": {
    "spoofing_tested": null,
    "attack_types": [],
    "liveness_detection": null,
    "defense_type": "none | heuristic | ML-based | null"
  },

  "usability": {
    "authentication_type": "explicit | implicit | continuous | null",
    "user_interaction_required": null,
    "latency": [{ "value": null, "unit": "" }],
    "energy_consumption": [{ "value": null, "unit": "" }],
    "usability_study": [], // list of study types (e.g., "SUS survey", "task-based evaluation", "longitudinal user study", NASA-TLX)
    "usability_issues": ""
  },

  "fusion": {
    "is_multimodal": null,
    "modalities_combined": [],
    "fusion_level": "feature | score | decision | hybrid | null",
    "benefit_over_unimodal": [{ "metric": "", "value": null, "unit": "" }]
  },

  "reproducibility": {
    "code_available": null,
    "datasets_all_public": null,
    "datasets_any_public": null,
    "benchmark_used": "",
    "reproducibility_score": "low | medium | high | null"
  },

  "insights": {
    "key_contribution": [],
    "main_strength": [],
    "main_weakness": [],
    "gap_category": [],
    "notes_for_review": ""
  }

  
}

======================
FINAL VALIDATION
======================
1. JSON MUST be valid (No trailing commas, no unclosed brackets, no quotes around arrays).
2. `exclusion_reason` is "" when relevance is "include".
3. Use floats for performance metrics (e.g., 1.5). Do NOT round to integers.
4. ONLY extract AUC, EER, FRR, FAR. Extract ONLY the single best result per metric found.
5. If spoofing is not tested, `security.spoofing_tested` must be false or null, even if standard FAR (zero-effort) is tested.
6. No text outside the JSON object.
7. When outputting null, use the JSON primitive null (without quotes). NEVER output the string "null".

Here is the paper: """

# --- Setup ---
def setup_directories():
    print("Setting up directories...")
    os.makedirs(PDF_FOLDER, exist_ok=True)
    os.makedirs(JSON_FOLDER, exist_ok=True)
    print(f"Folders '{PDF_FOLDER}' and '{JSON_FOLDER}' are ready.")


# --- PDF Processing ---
def extract_text_without_references(pdf_path):
    try:
        document = fitz.open(pdf_path)
        text = ""
        ref_keywords = ["references", "bibliography"]

        for page_num in range(len(document)):
            page = document.load_page(page_num)
            page_text = page.get_text()

            if any(re.search(r'\b' + keyword + r'\b', page_text.lower()) for keyword in ref_keywords):
                lower_text = page_text.lower()
                indices = [lower_text.find(keyword) for keyword in ref_keywords if keyword in lower_text]
                start_index = min(indices) if indices else None

                if start_index is not None:
                    text += page_text[:start_index].strip()
                    break

            text += page_text

        document.close()
        return text

    except Exception as e:
        print(f"ERROR reading PDF '{os.path.basename(pdf_path)}': {e}")
        return None


# --- JSON Extraction ---
def extract_json_from_response(raw_text):
    """Extract JSON from ```json code block or fallback to first JSON object."""
    
    # प्राथमिक method: ```json block
    match = re.search(r"```json\s*(.*?)\s*```", raw_text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()

    # fallback: first {...}
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if match:
        print("  - Fallback JSON extraction used.")
        return match.group(0).strip()

    print("  - ERROR: No JSON found in response.")
    return None


# --- API Call ---
def call_generative_model_with_retry(model, prompt):
    current_delay = API_RETRY_DELAY_SECONDS

    for attempt in range(API_RETRY_ATTEMPTS):
        try:
            print(f"  - API call (attempt {attempt + 1})...")
            response = model.generate_content(prompt)
            return response.text

        except Exception as e:
            print(f"  - ERROR: {e}")

            if attempt < API_RETRY_ATTEMPTS - 1:
                print(f"  - Retrying in {current_delay} seconds...")
                time.sleep(current_delay)
                current_delay *= 2
            else:
                print("  - All retries failed.")
                raise


# --- Main Pipeline ---
def main():
    if not API_KEY:
        print("Missing GOOGLE_API_KEY.")
        return

    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel(MODEL_NAME)

    setup_directories()

    pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.lower().endswith(".pdf")]

    if not pdf_files:
        print("No PDFs found.")
        return

    print(f"Found {len(pdf_files)} PDF(s).")
    print("-" * 50)

    for i, pdf_filename in enumerate(pdf_files):
        print(f"Processing {i + 1}/{len(pdf_files)}: {pdf_filename}")

        try:
            pdf_path = os.path.join(PDF_FOLDER, pdf_filename)

            # 1. Extract text
            article_text = extract_text_without_references(pdf_path)
            if not article_text:
                continue

            # 2. Send to model
            prompt = f"{SYSTEM_INSTRUCTION}\n\n--- PDF TEXT START ---\n{article_text}\n--- PDF TEXT END ---"
            raw_text = call_generative_model_with_retry(model, prompt)

            # 3. Extract JSON cleanly
            cleaned_text = extract_json_from_response(raw_text)
            if not cleaned_text:
                print("  - Skipping due to missing JSON.")
                continue

            # 4. Parse JSON
            try:
                json_data = json.loads(cleaned_text)
            except json.JSONDecodeError as e:
                print(f"  - JSON decode error: {e}")
                print("  - Extracted text:", cleaned_text)
                continue

            # 5. Save JSON (same filename as PDF)
            json_filename = os.path.splitext(pdf_filename)[0] + ".json"
            json_path = os.path.join(JSON_FOLDER, json_filename)

            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(json_data, f, indent=2, ensure_ascii=False)

            print(f"  - Saved: {json_filename}")

        except Exception as e:
            print(f"Unhandled error: {e}")

        finally:
            print(f"Waiting {REQUEST_DELAY_SECONDS} seconds...")
            time.sleep(REQUEST_DELAY_SECONDS)
            print("-" * 50)

    print("All PDFs processed.")


if __name__ == "__main__":
    main()
